# Image-level hazard references

Default execution prepares 11 explicitly synthetic CPU fixture images in separate data/run namespaces. Real data and downloads require the opt-in switches below. References follow the frozen [hazard protocol](../docs/hazard_protocol.md); they never enter VLM inference. No model, GPU or visualization action is included.


In [ ]:
from pathlib import Path
import copy
import json
import os
import sys

# Configuration: explicit values override environment variables.
REPO_OVERRIDE = None  # Checkout or VLM_evaluation directory; accepts Windows/Linux paths.
DATA_ROOT = None
RUN_ROOT = None
CACHE_ROOT = None
RUN_FIXTURE = True
RUN_REAL_PREPARATION = False
DOWNLOAD_COCO = False  # Downloads annotation archive and only frozen selected RGB IDs.
RELLIS_SOURCE_DIR = None  # Existing extracted original RGB/semantic-ID tree.


def locate_component(override=None):
    selected = override or os.environ.get("TRAVERSABILITY_REPO_ROOT")
    start = Path(selected).expanduser().resolve() if selected else Path.cwd().resolve()
    bases = (start,) if selected else (start, *start.parents)
    candidates = [p for base in bases for p in (base, base / "VLM_evaluation")]
    for candidate in candidates:
        if (candidate / "docs/hazard_interfaces.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot locate VLM_evaluation; set REPO_OVERRIDE or TRAVERSABILITY_REPO_ROOT."
    )


REPO = locate_component(REPO_OVERRIDE)


def resolve_root(explicit, variable, default):
    path = Path(explicit or os.environ.get(variable) or default).expanduser()
    return (path if path.is_absolute() else REPO / path).resolve()


DATA_ROOT = resolve_root(DATA_ROOT, "TRAVERSABILITY_DATA_ROOT", "data")
RUN_ROOT = resolve_root(RUN_ROOT, "TRAVERSABILITY_RUN_ROOT", "runs")
CACHE_ROOT = resolve_root(CACHE_ROOT, "TRAVERSABILITY_CACHE_ROOT", ".cache")
sys.path.insert(0, str(REPO / "src"))
from traversability_hazard_data import prepare_run, validate_run

CONFIG = json.loads((REPO / "configs/hazards/data.json").read_text(encoding="utf-8"))
CONFIG.update(repo_root=str(REPO), data_root=str(DATA_ROOT),
              run_root=str(RUN_ROOT), cache_root=str(CACHE_ROOT))
CONFIG["rellis"]["source_dir"] = RELLIS_SOURCE_DIR
print({"data_root": str(DATA_ROOT), "run_root": str(RUN_ROOT),
       "cache_root": str(CACHE_ROOT), "real_preparation": RUN_REAL_PREPARATION,
       "download_coco": DOWNLOAD_COCO})


## Synthetic CPU check

The fixture contains five RELLIS-schema frames (one per sequence) and six COCO-schema images (four target-positive/two target-negative). It checks source decoding and references, not real experiment coverage. All notebook preparation uses public APIs.


In [ ]:
FIXTURE_RUN = None
FIXTURE_CHECK = None
if RUN_FIXTURE:
    FIXTURE_CONFIG = copy.deepcopy(CONFIG)
    FIXTURE_CONFIG.update(
        run_name="hazard_prompt_v1_fixture",
        data_root=str(DATA_ROOT / "fixtures/hazard_prompt_v1"),
    )
    FIXTURE_RUN = prepare_run(FIXTURE_CONFIG, fixture=True)
    FIXTURE_CHECK = validate_run(FIXTURE_RUN, FIXTURE_CONFIG["data_root"])
    print({"fixture": FIXTURE_CHECK["fixture"], "counts": FIXTURE_CHECK["counts"],
           "coverage": FIXTURE_CHECK["coverage"]})


## Optional real-data actions

RELLIS reuses existing RGB/original semantic-ID files; see [source links and checks](../docs/hazard_data.md). It is never fetched here. Real preparation freezes 100 RELLIS plus 40 COCO selections when sources support the recipe; missing inputs remain pending. COCO downloads require the explicit switch above and use annotation-selected IDs only. Keep the same inputs to resume; use a new run name when completed inputs change.


In [ ]:
REAL_RUN = None
REAL_CHECK = None
if DOWNLOAD_COCO:
    from traversability_hazard_data import (
        download_coco_annotations, download_selected_coco_images,
    )
    download_coco_annotations(CONFIG, allow_download=True)
    REAL_RUN = prepare_run(CONFIG)  # Freeze annotation-selected IDs before RGB download.
    download_selected_coco_images(
        REAL_RUN, DATA_ROOT, CACHE_ROOT, allow_download=True,
    )
    REAL_RUN = prepare_run(CONFIG)
    REAL_CHECK = validate_run(REAL_RUN, DATA_ROOT)


In [ ]:
if RUN_REAL_PREPARATION:
    REAL_RUN = prepare_run(CONFIG)
    REAL_CHECK = validate_run(REAL_RUN, DATA_ROOT)


## Inspect coverage

Inspect source/split counts and missing strata before inference. Fixture checks remain explicitly marked synthetic. Phone clips, cables, independent spill/floor-contact cases and physical safety remain pending optional inputs.


In [ ]:
for label, run_dir, check in (
    ("synthetic_fixture", FIXTURE_RUN, FIXTURE_CHECK),
    ("real_inputs", REAL_RUN, REAL_CHECK),
):
    if check is not None:
        print(label, str(run_dir))
        print(json.dumps({"fixture": check["fixture"], "counts": check["counts"],
                          "coverage": check["coverage"],
                          "dataset_fingerprint": check["dataset_fingerprint"]},
                         indent=2, sort_keys=True))

# Freeze the diagnostic test/warmup selection after all intended real inputs exist.
SAM_SELECTION = None
if REAL_CHECK is not None and REAL_CHECK["coverage"]["complete_planned_frame_coverage"]:
    from traversability_hazard_segmentation import freeze_selection
    SAM_SELECTION = freeze_selection(REAL_RUN, data_root=DATA_ROOT,
                                     policy_path=CONFIG["policy_path"])
    print({"selection_frozen": SAM_SELECTION["complete"],
           "selection_hash": SAM_SELECTION["selection_hash"]})
